# seccal-12b: fine-tune LFM2.5-1.2B-Instruct

**Before running:** `Runtime → Change runtime type → T4 GPU` (free tier is enough).

Then `Runtime → Run all`. Embedded: training script + 399 train / 52 val rows. Nothing else to upload. Output GGUFs are copied to your Google Drive at `MyDrive/seccal-12b/` (you will get one Drive authorization prompt). Total time on a T4 is roughly 10–20 minutes including installs.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

## 1. Unpack the embedded training script and data

In [ ]:
import base64, io, zipfile, os
BUNDLE_B64 = '''...'''
z = zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64)))
z.extractall('/content/seccal')
os.chdir('/content/seccal')
for n in z.namelist(): print(n, os.path.getsize(n), 'bytes')

## 2. Install the training stack (a few minutes)

In [ ]:
%pip install -q -U unsloth "trl>=0.15" datasets transformers
import unsloth, trl, transformers
print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| transformers', transformers.__version__)

## 3. (Optional) Hugging Face token

Only needed if the base model download asks for one. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and this cell will pick it up; otherwise it is skipped.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets')
except Exception:
    print('no HF_TOKEN secret; continuing without one')

## 4. Train, then export Q4_K_M and Q8_0 GGUFs

In [ ]:
!python train/sft_unsloth.py --train data/sft/train.jsonl --val data/sft/val.jsonl --out out/seccal-12b --epochs 3
!ls -la out/seccal-12b/gguf/

## 5. Save the GGUFs to Google Drive

Approve the Drive prompt. Files land in `MyDrive/seccal-12b/`. Then on your Mac, copy them into the repo's `models/` folder.

In [ ]:
from google.colab import drive
import shutil, glob, os
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/seccal-12b'
os.makedirs(dst, exist_ok=True)
for f in glob.glob('out/seccal-12b/gguf/*.gguf'):
    shutil.copy(f, dst)
    print('copied', os.path.basename(f), round(os.path.getsize(f)/1e6), 'MB')
shutil.copy('out/seccal-12b/train_stats.json', dst)
print('done ->', dst)

### Fallback: direct browser download

If Drive is not an option, run this instead (two downloads, ~0.7 GB and ~1.3 GB).

In [ ]:
from google.colab import files
import glob
for f in glob.glob('out/seccal-12b/gguf/*.gguf'):
    files.download(f)